In [ ]:
# Cell 1 – Installation & Version Check
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["einops", "timm", "fvcore"]:
    install(pkg)

import torch, torchvision, numpy as np, albumentations as A
import PIL, matplotlib, sklearn, einops, timm

print("\n=== Library Versions ===")
for name, mod in [("torch", torch), ("torchvision", torchvision),
                   ("numpy", np), ("albumentations", A),
                   ("einops", einops), ("PIL", PIL), ("timm", timm)]:
    print(f"  {name}: {getattr(mod, '__version__', 'n/a')}")

print(f"\ntorch.cuda.is_available(): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# Cell 2 – Imports & Reproducibility
import os, random, json, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score

import einops
import timm

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"🌱 Seed set to {seed}")

print("Imports complete.")


In [ ]:
# Cell 3 – Configuration
# ═══════════════════════════════════════════════════════════════════════════════
from dataclasses import dataclass, field
import torch

@dataclass
class CFG_S1:
    """LightVessel-Net v11 – Cross-Dataset Training & Evaluation."""
    #IMAGE_SIZE      : int   = 512
    BATCH_SIZE      : int   = 16
    EPOCHS          : int   = 1
    LR              : float = 8e-3
    WEIGHT_DECAY    : float = 5e-4
    PATIENCE        : int   = 75
    GRAD_CLIP       : float = 1.5
    MC_SAMPLES      : int   = 12
    DROPOUT_RATE    : float = 0.05
    DROPBLOCK_SIZE  : int   = 5
    AUX_WEIGHT      : float = 0.2
    USE_TTA         : bool  = False
    MIXUP_ALPHA     : float = 0.2
    CUTMIX_PROB     : float = 0.2
    MODEL_SAVE_PATH : str   = "/kaggle/working/best_lightvesselnet_v6.pth"
    FOLD_SAVE_DIR   : str   = "/kaggle/working/cv_folds"
    Loss_alpha      : float = 0.7
    Loss_gamma      : float = 2.75


@dataclass
class CFG_SHARED:
    DATASET_ROOT : str = "/kaggle/input/datasets/shadmansobhan/fivesvascular-evaluation-and-segmentation"
    TRAIN_SPLIT  : float = 0.8
    SEED         : int   = 42
    DEBUG        : bool  = True
    DEVICE       : torch.device = field(
        default_factory=lambda: torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"))

CFG_S1 = CFG_S1()
CFG    = CFG_SHARED()

# Aliases expected by downstream cells
#CFG.IMAGE_SIZE         = CFG_S1.IMAGE_SIZE
CFG.MC_SAMPLES         = CFG_S1.MC_SAMPLES
CFG.DROPOUT_RATE       = CFG_S1.DROPOUT_RATE
CFG.MODEL_SAVE_PATH_S1 = CFG_S1.MODEL_SAVE_PATH

def set_seed(seed):
    import random, numpy as np, torch
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"🌱 Seed set to {seed}")

set_seed(CFG.SEED)
print("CFG_S1:", CFG_S1)
print("CFG_SHARED:", CFG)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 4 — Preprocessing
# ═══════════════════════════════════════════════════════════════════════════════

import cv2
import numpy as np

PREPROCESSING_MODE: int = 6

PREPROCESSING_NAMES = {
    1: "No Pre-processing",
    2: "Gamma Transformation (γ=0.7)",
    3: "Histogram Equalization",
    4: "CLAHE (clip=4, tile=8×8)",
    5: "Contrast Stretching",
    6: "Green Channel + CLAHE",
    7: "BG-corrected Weighted Fusion + Tuned CLAHE",
}
print(f"✅ Pre-processing mode: [{PREPROCESSING_MODE}] {PREPROCESSING_NAMES[PREPROCESSING_MODE]}")


def preproc_none(image):
    return image


def preproc_gamma(image, gamma=0.7):
    inv_gamma = 1.0 / gamma
    table = np.array(
        [((i / 255.0) ** inv_gamma) * 255 for i in range(256)], dtype=np.uint8
    )
    return cv2.LUT(image, table)


def preproc_histeq(image):
    ycrcb = cv2.cvtColor(image, cv2.COLOR_RGB2YCrCb)
    ycrcb[:, :, 0] = cv2.equalizeHist(ycrcb[:, :, 0])
    return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)


def preproc_clahe(image, clip_limit=4.0, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def preproc_contrast_stretch(image, low_pct=2.0, high_pct=98.0):
    out = np.empty_like(image)
    for c in range(image.shape[2]):
        ch = image[:, :, c].astype(np.float32)
        lo, hi = np.percentile(ch, low_pct), np.percentile(ch, high_pct)
        ch = (
            np.clip((ch - lo) / (hi - lo + 1e-8) * 255, 0, 255)
            if hi > lo
            else np.clip(ch, 0, 255)
        )
        out[:, :, c] = ch.astype(np.uint8)
    return out


def preproc_green_clahe(image, clip_limit=4.0, tile_grid_size=(8, 8)):
    """Mode 6 — original green channel + CLAHE (kept as baseline)."""
    green = image[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    green_clahe = clahe.apply(green)
    return cv2.merge([green_clahe, green_clahe, green_clahe])


# ── Mode 7 — BG-corrected weighted fusion + tuned CLAHE ──────────────────────

def _bg_correct(channel_f32: np.ndarray, sigma: float = 20.0) -> np.ndarray:
    """
    Subtract a large-scale Gaussian blur to remove the retinal illumination
    gradient (bright centre, dark periphery). Adding 128 re-centres intensity.

    sigma=20 at 512×512 covers ~8% of image width — empirically good for
    fundus background. Scale sigma proportionally if you change input resolution:
        sigma = 20 * (input_size / 512)
    """
    blur = cv2.GaussianBlur(channel_f32, (0, 0), sigmaX=sigma)
    return np.clip(channel_f32 - blur + 128.0, 0.0, 255.0)


def preproc_mode7(
    image,
    # Fusion weights — green dominates, red adds optic-disc structure,
    # blue is noisiest in fundus so gets minimal weight.
    w_r: float = 0.07,
    w_g: float = 0.88,
    w_b: float = 0.05,
    # Background correction
    bg_sigma: float = 20.0,
    # CLAHE — smaller tiles (16×16 → 32px at 512 input) stay closer to
    # vessel diameter scale; lower clip reduces noise amplification.
    clip_limit: float = 2.5,
    tile_grid_size: tuple = (16, 16),
):
    """
    Mode 7 pipeline (all steps in float32 until the final uint8 cast):

      1. Weighted RGB → single luminance channel
            Weights tuned for retinal fundus: green carries the most
            vessel/background contrast-to-noise ratio (CNR); red preserves
            optic disc structure; blue contributes noise more than signal.

      2. Background field correction (Gaussian subtraction)
            Removes the low-frequency illumination gradient that causes
            CLAHE to waste contrast budget on background variation rather
            than vessel edges.

      3. CLAHE with tuned tile size
            tile_grid_size=(16,16) → 32×32px tiles at 512 input.
            Vessel cross-sections are ~3–8px wide, so a 32px tile still
            treats them as local features (not background). clip_limit=2.5
            is gentler than mode-6's 4.0 — the bg-correction step already
            does the heavy lifting, so aggressive clipping adds noise here.

      4. Replicate to 3-channel output so the rest of the pipeline
         (normalization, augmentation, model input) stays unchanged.

    Hyperparameters to grid-search first: clip_limit ∈ {2.0, 2.5, 3.0},
    tile_grid_size ∈ {(8,8), (12,12), (16,16)}, bg_sigma ∈ {15, 20, 25}.
    """
    r = image[:, :, 0].astype(np.float32)
    g = image[:, :, 1].astype(np.float32)
    b = image[:, :, 2].astype(np.float32)

    # Step 1 — weighted luminance fusion
    fused = np.clip(w_r * r + w_g * g + w_b * b, 0.0, 255.0)

    # Step 2 — background field correction
    fused = _bg_correct(fused, sigma=bg_sigma)

    # Step 3 — CLAHE
    fused_u8 = fused.astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    out = clahe.apply(fused_u8)

    # Step 4 — 3-channel output
    return cv2.merge([out, out, out])


# ── Dispatch table ────────────────────────────────────────────────────────────

_PREPROC_FN = {
    1: preproc_none,
    2: preproc_gamma,
    3: preproc_histeq,
    4: preproc_clahe,
    5: preproc_contrast_stretch,
    6: preproc_green_clahe,
    7: preproc_mode7,
}


def apply_preprocessing(image: np.ndarray) -> np.ndarray:
    """
    Entry point for the rest of the pipeline.
    Always receives uint8 RGB (H, W, 3) and returns the same dtype/shape.
    """
    fn = _PREPROC_FN.get(PREPROCESSING_MODE)
    if fn is None:
        raise ValueError(
            f"Unknown PREPROCESSING_MODE={PREPROCESSING_MODE}. "
            f"Valid options: {list(_PREPROC_FN.keys())}"
        )
    out = fn(image)
    assert out.shape == image.shape, (
        f"Preprocessor changed shape: {image.shape} → {out.shape}"
    )
    assert out.dtype == np.uint8, (
        f"Preprocessor returned {out.dtype}, expected uint8"
    )
    return out


# ── Smoke test ────────────────────────────────────────────────────────────────

_dummy = np.random.randint(0, 256, (512, 512, 3), dtype=np.uint8)
_out   = apply_preprocessing(_dummy)
assert _out.shape == _dummy.shape and _out.dtype == np.uint8
print(f"✅ apply_preprocessing OK — mode {PREPROCESSING_MODE}: "
      f"{PREPROCESSING_NAMES[PREPROCESSING_MODE]}")
del _dummy, _out

In [ ]:
# Cell 5 – Dataset Scanner
SUPPORTED_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def scan_dataset(root):
    root = Path(root); results = {}
    for split in ("train", "test"):
        img_dir  = root / split / "images"
        mask_dir = root / split / "masks"
        pairs = []
        if not img_dir.exists():
            print(f"⚠️  {img_dir} not found – skipping {split}")
            results[split] = pairs; continue
        for img_path in sorted(img_dir.iterdir()):
            if img_path.suffix.lower() not in SUPPORTED_EXTS: continue
            stem = img_path.stem.lower(); mask_path = None
            for ext in SUPPORTED_EXTS:
                for cand in [mask_dir/(img_path.stem+ext), mask_dir/(stem+ext)]:
                    if cand.exists(): mask_path = cand; break
                if mask_path: break
            if mask_path is None:
                raise FileNotFoundError(f"❌ Mask not found for '{img_path}'")
            pairs.append((img_path, mask_path))
        results[split] = pairs
    train_pairs = results.get("train", [])
    test_pairs  = results.get("test",  [])
    if CFG.DEBUG:
        print(f"\n📂 Dataset – root: {root}")
        print(f"  Train: {len(train_pairs)},  Test: {len(test_pairs)}")
    return train_pairs, test_pairs

train_pairs, test_pairs = scan_dataset(CFG.DATASET_ROOT)
random.shuffle(train_pairs)
n_train     = int(len(train_pairs) * CFG.TRAIN_SPLIT)
val_pairs   = train_pairs[n_train:]
train_pairs = train_pairs[:n_train]
print(f"After split → train: {len(train_pairs)}, val: {len(val_pairs)}, test: {len(test_pairs)}")


In [ ]:
# Cell 6 – Augmentation Pipeline  (v6: native resolution, no resize + MixUp/CutMix)
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_train_transform():
    extra_contrast = []
    if PREPROCESSING_MODE != 4:
        extra_contrast = [A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=0.6)]
    # NOTE: No A.Resize — images are used at their native resolution.
    # All images/masks in the dataset must be the same size.
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=30, p=0.5),
        A.ElasticTransform(alpha=120, sigma=6, p=0.3),
        A.GridDistortion(p=0.3),
        A.OpticalDistortion(distort_limit=0.2, p=0.2),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        A.Sharpen(alpha=(0.1, 0.3), lightness=(0.9, 1.1), p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
        A.GaussNoise(var_limit=(5.0, 40.0), p=0.4),
        A.GaussianBlur(blur_limit=(3,5), p=0.2),
        *extra_contrast,
        A.CoarseDropout(max_holes=6, max_height=16, max_width=16, p=0.25),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

def get_val_transform():
    extra_contrast = []
    if PREPROCESSING_MODE != 4:
        extra_contrast = [A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=1.0)]
    # NOTE: No A.Resize — images are used at their native resolution.
    return A.Compose([
        *extra_contrast,
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

# ── MixUp / CutMix helpers (applied at batch level in training loop) ──────────
import torch

def mixup_batch(imgs, masks, alpha=0.2):
    """MixUp: blend two images & masks with a Beta-sampled λ."""
    if alpha <= 0:
        return imgs, masks
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return lam * imgs + (1 - lam) * imgs[idx], lam * masks + (1 - lam) * masks[idx]

def cutmix_batch(imgs, masks, prob=0.2):
    """CutMix: paste a random rectangular region from another image."""
    if prob <= 0 or np.random.rand() > prob:
        return imgs, masks
    B, C, H, W = imgs.shape
    lam = float(np.random.beta(1.0, 1.0))
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, x2 = max(cx - cut_w // 2, 0), min(cx + cut_w // 2, W)
    y1, y2 = max(cy - cut_h // 2, 0), min(cy + cut_h // 2, H)
    idx = torch.randperm(B, device=imgs.device)
    imgs  = imgs.clone();  imgs[:, :, y1:y2, x1:x2]  = imgs[idx, :, y1:y2, x1:x2]
    masks = masks.clone(); masks[:, :, y1:y2, x1:x2] = masks[idx, :, y1:y2, x1:x2]
    return imgs, masks

print("Augmentation pipelines ready.")


In [ ]:
# Cell 7 – Whole-Image Dataset & DataLoader  

class RetinalImageDataset(Dataset):
    def __init__(self, pairs, mode="train"):
        self.mode      = mode
        self.transform = get_train_transform() if mode == "train" else get_val_transform()
        self.pairs     = pairs
        first_img = Image.open(pairs[0][0]).convert("RGB") if pairs else None
        native_size = first_img.size if first_img is not None else ("?", "?")
        print(f"  [{mode}] {len(pairs)} whole images loaded at native resolution {native_size[0]}×{native_size[1]} (no resize)")

    def __len__(self):  return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        image = apply_preprocessing(image)
        mask  = (np.array(Image.open(mask_path).convert("L")) > 127).astype(np.uint8)
        out   = self.transform(image=image, mask=mask)
        return {"image": out["image"].float(),
                "mask":  out["mask"].float().unsqueeze(0)}

train_dataset = RetinalImageDataset(train_pairs, mode="train")
val_dataset   = RetinalImageDataset(val_pairs,   mode="val")

N_WORKERS = min(4, os.cpu_count() or 2)
train_loader = DataLoader(train_dataset, batch_size=CFG_S1.BATCH_SIZE,
                          shuffle=True, num_workers=N_WORKERS,
                          pin_memory=True, drop_last=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset, batch_size=CFG_S1.BATCH_SIZE,
                          shuffle=False, num_workers=N_WORKERS,
                          pin_memory=True, persistent_workers=True)

# Quick shape sanity-check
for b in train_loader:
    print(f"Train batch — image: {b['image'].shape}, mask: {b['mask'].shape}")
    break
print("DataLoaders ready.")


# Model

In [ ]:
# Cell 8 – Define Model
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Helpers ───────────────────────────────────────────────────────────────────

def make_norm(channels: int, preferred_groups: int = 8) -> nn.Module:
    """GroupNorm with graceful fallback for small channel counts."""
    for g in range(preferred_groups, 0, -1):
        if channels % g == 0:
            return nn.GroupNorm(g, channels)
    return nn.GroupNorm(1, channels)


class DropBlock2d(nn.Module):
    """Structured spatial dropout — drops contiguous blocks."""
    def __init__(self, block_size: int = 5, keep_prob: float = 0.9):
        super().__init__()
        self.block_size = block_size
        self.keep_prob  = keep_prob

    def forward(self, x):
        if not self.training or self.keep_prob == 1.0:
            return x
        B, C, H, W = x.shape
        p = (1 - self.keep_prob) / (self.block_size ** 2)
        seed = torch.bernoulli(torch.full((B, C, H, W), p, device=x.device))
        pad  = self.block_size // 2
        mask = F.max_pool2d(
            F.pad(seed, [pad]*4),
            kernel_size=(self.block_size, self.block_size), stride=1)[:, :, :H, :W]
        keep = 1.0 - mask
        return x * keep * (keep.numel() / (keep.sum() + 1e-6))


# ── MicroBlock with cheap SE ─────────────────────────────────────────────────
class MicroBlockSE(nn.Module):
    """
    Depthwise-Separable block with Squeeze-and-Excitation.
    DW-3×3 → GN → SiLU → PW-1×1 → GN → SiLU + SE + residual.
    """
    def __init__(self, in_ch: int, out_ch: int,
                 block_size: int = 5, keep_prob: float = 0.9, se_ratio: int = 32):
        super().__init__()
        self.dw   = nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch, bias=False)
        self.bn1  = make_norm(in_ch)
        self.drop = DropBlock2d(block_size, keep_prob)
        self.pw   = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2  = make_norm(out_ch)
        self.act  = nn.SiLU(inplace=True)
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        
        mid = max(out_ch // se_ratio, 4)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_ch, mid, 1, bias=False),
            nn.SiLU(inplace=True),
            nn.Conv2d(mid, out_ch, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        out = self.act(self.bn1(self.dw(x)))
        out = self.drop(out)
        out = self.act(self.bn2(self.pw(out)))
        out = out * self.se(out)
        return out + self.skip(x)


# ── PatchEmbed stem ───────────────────────────────────────────────────────────
class PatchEmbed(nn.Module):
    def __init__(self, in_ch: int = 3, out_ch: int = 18):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=2, stride=2, bias=False)
        self.norm = make_norm(out_ch)

    def forward(self, x):
        return self.norm(self.proj(x))


# ── EfficientBottleneck (DW-5×5 + SE + PW) ───────────────────────────────────
class EfficientBottleneck(nn.Module):
    def __init__(self, in_ch: int, out_ch: int,
                 block_size: int = 5, keep_prob: float = 0.9, se_ratio: int = 32):
        super().__init__()
        mid = max(in_ch // se_ratio, 4)
        self.dw    = nn.Conv2d(in_ch, in_ch, 5, padding=2, groups=in_ch, bias=False)
        self.bn_dw = make_norm(in_ch)
        self.drop  = DropBlock2d(block_size, keep_prob)
        self.se1   = nn.Linear(in_ch, mid, bias=False)
        self.se2   = nn.Linear(mid, in_ch, bias=False)
        self.pw    = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn_pw = make_norm(out_ch)
        self.act   = nn.SiLU(inplace=True)
        self.skip  = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.act(self.bn_dw(self.dw(x)))
        out = self.drop(out)
        se  = self.act(self.se1(out.mean(dim=[2, 3])))
        se  = torch.sigmoid(self.se2(se)).unsqueeze(-1).unsqueeze(-1)
        out = self.act(self.bn_pw(self.pw(out * se)))
        return out + self.skip(x)


# ── MSFA v2 — Multi-Scale Feature Aggregation ────────────────────────────────
class MSFA(nn.Module):
    """
    4-branch multi-scale fusion with quarter-channel split.
    Branches: 1×1 (local), DW-3×3 d=1, d=2, d=4.
    """
    def __init__(self, channels: int, se_ratio: int = 32):
        super().__init__()
        assert channels % 4 == 0, f"channels must be divisible by 4, got {channels}"
        mid = max(channels // se_ratio, 4)
        q = channels // 4
        
        self.b0 = nn.Sequential(
            nn.Conv2d(channels, q, 1, bias=False),
            make_norm(q), nn.SiLU(inplace=True))
        self.b1 = nn.Sequential(
            nn.Conv2d(channels, q, 3, padding=1, dilation=1, groups=q, bias=False),
            make_norm(q), nn.SiLU(inplace=True))
        self.b2 = nn.Sequential(
            nn.Conv2d(channels, q, 3, padding=2, dilation=2, groups=q, bias=False),
            make_norm(q), nn.SiLU(inplace=True))
        self.b3 = nn.Sequential(
            nn.Conv2d(channels, q, 3, padding=4, dilation=4, groups=q, bias=False),
            make_norm(q), nn.SiLU(inplace=True))
        
        self.fuse = nn.Sequential(
            nn.Conv2d(channels, channels, 1, bias=False),
            make_norm(channels), nn.SiLU(inplace=True))
        
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.SiLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        feat = torch.cat([self.b0(x), self.b1(x), self.b2(x), self.b3(x)], dim=1)
        out  = self.fuse(feat)
        out  = out * self.se(out)
        return out + x


# ── PixelShuffle upsample ─────────────────────────────────────────────────────
class PixelShuffleUp(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch * 4, 1, bias=False)
        self.norm = make_norm(out_ch * 4)
        self.ps   = nn.PixelShuffle(2)

    def forward(self, x):
        return self.ps(self.norm(self.proj(x)))


# ── Spatial Attention (channel-agnostic, ultra-cheap) ─────────────────────────
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_val, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg, max_val], dim=1)
        return x * self.sigmoid(self.conv(concat))

# ── Spatial size alignment helper ────────────────────────────────────────────
def _pad_to_match(x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    Pad `x` (upsampled decoder feature) to exactly match the spatial dims
    of `target` (encoder skip connection).  Off-by-one mismatches arise
    whenever the input H or W is not divisible by the total stride (8×),
    causing floor-division at each pooling stage to lose a pixel.
    We use right/bottom padding with reflection so no hard edge is introduced.
    """
    dh = target.shape[2] - x.shape[2]   # height difference (≥0 after upsample)
    dw = target.shape[3] - x.shape[3]   # width  difference (≥0 after upsample)
    if dh == 0 and dw == 0:
        return x
    # F.pad order: (left, right, top, bottom)
    return F.pad(x, (0, dw, 0, dh), mode="reflect")


class LightVesselNet(nn.Module):
    def __init__(self, dropout_rate=0.05, dropblock_size=5):
        super().__init__()
        kp = 1.0 - dropout_rate
        bs = dropblock_size

        self.stem = PatchEmbed(3, 18)
        self.enc1 = MicroBlockSE(18, 26, bs, kp, se_ratio=32)
        self.enc2 = MicroBlockSE(26, 40, bs, kp, se_ratio=32)
        self.enc3 = EfficientBottleneck(40, 60, bs, kp, se_ratio=32)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = MSFA(60, se_ratio=32)

        self.up3 = PixelShuffleUp(60, 60)
        self.up2 = PixelShuffleUp(60, 40)
        self.up1 = PixelShuffleUp(40, 26)

        self.dec3 = MicroBlockSE(120, 60, bs, kp, se_ratio=32)
        self.dec2 = MicroBlockSE(80, 40, bs, kp, se_ratio=32)
        self.dec1 = MicroBlockSE(52, 26, bs, kp, se_ratio=32)

        self.sa_bottleneck = SpatialAttention(7)
        self.edge_proj = nn.Sequential(
            nn.Conv2d(18, 26, 1, bias=False),
            make_norm(26),
            nn.SiLU(inplace=True)
        )

        self.out_conv  = nn.Conv2d(26, 1, 1)
        self.aux_head3 = nn.Conv2d(60, 1, 1)
        self.aux_head2 = nn.Conv2d(40, 1, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.GroupNorm, nn.BatchNorm2d)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')

    def forward(self, x, return_aux=False):
        stem = self.stem(x)
        s1   = self.enc1(stem)
        s2   = self.enc2(self.pool(s1))
        s3   = self.enc3(self.pool(s2))
        b    = self.bottleneck(self.pool(s3))
        b    = self.sa_bottleneck(b)

        d3 = self.dec3(torch.cat([_pad_to_match(self.up3(b),  s3), s3], dim=1))
        d2 = self.dec2(torch.cat([_pad_to_match(self.up2(d3), s2), s2], dim=1))
        d1 = self.dec1(torch.cat([_pad_to_match(self.up1(d2), s1), s1], dim=1))

        edge = self.edge_proj(stem)
        d1 = d1 + _pad_to_match(edge, d1)   # also guard edge + d1 residual

        # Upsample back to input resolution (handles non-stride-8 input sizes)
        d1_full = F.interpolate(d1, size=(x.shape[2], x.shape[3]),
                                mode='bilinear', align_corners=False)
        main = self.out_conv(d1_full)

        if return_aux:
            return main, self.aux_head3(d3), self.aux_head2(d2)
        return main

# ── Example usage / self-test ─────────────────────────────────────────────────
if __name__ == "__main__":
    import torch
    
    model = LightVesselNet(dropout_rate=CFG_S1.DROPOUT_RATE, dropblock_size=CFG_S1.DROPBLOCK_SIZE)
    x = torch.randn(2, 3, 512, 512)
    
    with torch.no_grad():
        main, aux3, aux2 = model(x, return_aux=True)
        print(f"Main: {main.shape}")
        print(f"Aux3: {aux3.shape}")
        print(f"Aux2: {aux2.shape}")
    
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Params: {n_params/1e6:.3f}M")

In [ ]:
# Cell 9 – Loss Function

class VesselLoss(nn.Module):

    def __init__(self, alpha = CFG_S1.Loss_alpha, focal_gamma = CFG_S1.Loss_gamma,
                 aux_weight = CFG_S1.AUX_WEIGHT):
        super().__init__()
        self.alpha      = alpha          # FN weight in Tversky
        self.beta       = 1 - alpha      # FP weight
        self.gamma      = focal_gamma
        self.aux_weight = aux_weight

    def tversky_focal(self, logits, targets, eps: float = 1e-6):
        probs = torch.sigmoid(logits)
        # Resize targets to match logits if aux head
        if probs.shape != targets.shape:
            targets = F.interpolate(targets.float(), size=probs.shape[2:],
                                    mode='nearest')
        tp = (probs * targets).sum(dim=[1, 2, 3])
        fn = ((1 - probs) * targets).sum(dim=[1, 2, 3])
        fp = (probs * (1 - targets)).sum(dim=[1, 2, 3])
        tversky = (tp + eps) / (tp + self.alpha * fn + self.beta * fp + eps)
        tversky_loss = (1 - tversky).mean()

        # Focal BCE
        bce  = F.binary_cross_entropy_with_logits(logits, targets.float(),
                                                   reduction='none')
        pt   = torch.exp(-bce)
        focal = ((1 - pt) ** self.gamma * bce).mean()
        return tversky_loss + focal

    def forward(self, main_logits, aux3_logits, aux2_logits, targets):
        main_loss = self.tversky_focal(main_logits, targets)
        aux3_loss = self.tversky_focal(aux3_logits, targets)
        aux2_loss = self.tversky_focal(aux2_logits, targets)
        return main_loss + self.aux_weight * (aux3_loss + aux2_loss)


criterion = VesselLoss()

In [ ]:
# Cell 10 – Evaluation Metrics

from sklearn.metrics import roc_auc_score, average_precision_score

def compute_metrics(pred_prob, gt, threshold=0.5):
    pred_bin = (pred_prob >= threshold).astype(np.uint8).ravel()
    gt_flat  = gt.ravel().astype(np.uint8)
    TP = int(((pred_bin==1)&(gt_flat==1)).sum())
    FP = int(((pred_bin==1)&(gt_flat==0)).sum())
    FN = int(((pred_bin==0)&(gt_flat==1)).sum())
    TN = int(((pred_bin==0)&(gt_flat==0)).sum())
    eps = 1e-8
    se  = TP / (TP + FN + eps);   sp  = TN / (TN + FP + eps)
    acc = (TP + TN) / (TP + TN + FP + FN + eps)
    f1  = 2*TP / (2*TP + FP + FN + eps)
    iou = TP / (TP + FP + FN + eps)
    try:    auc = float(roc_auc_score(gt_flat, pred_prob.ravel()))
    except: auc = 0.5
    try:    pr  = float(average_precision_score(gt_flat, pred_prob.ravel()))
    except: pr  = 0.0
    return {"se": float(se), "sp": float(sp), "acc": float(acc),
            "auc": float(auc), "pr": float(pr), "f1": float(f1), "iou": float(iou)}

def compute_ece(pred_prob, gt, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for lo, hi in zip(bin_boundaries[:-1], bin_boundaries[1:]):
        mask = (pred_prob >= lo) & (pred_prob < hi)
        if mask.sum() == 0: continue
        ece += mask.sum() / pred_prob.size * abs(gt[mask].mean() - pred_prob[mask].mean())
    return float(ece)


def compute_flops(model, input_size, device):
    try:
        from fvcore.nn import FlopCountAnalysis
        dummy = torch.randn(1, 3, *input_size).to(device)
        model = model.to(device)  # ← already present in fvcore path
        model.eval()
        with torch.no_grad():
            return FlopCountAnalysis(model, dummy).total() / 1e9
    except Exception:
        pass
    
    # Fallback: manual conv hook
    total_flops = [0]
    hooks = []
    def conv_hook(module, inp, out):
        b, c_out, h, w = out.shape
        c_in = inp[0].shape[1]
        kH, kW = module.kernel_size
        total_flops[0] += 2*b*c_out*h*w*(c_in//module.groups)*kH*kW
    
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
    
    dummy = torch.randn(1, 3, *input_size).to(device)
    model = model.to(device)  # ← ADD THIS LINE (was missing!)
    model.eval()
    with torch.no_grad(): 
        model(dummy)
    
    for h in hooks: 
        h.remove()
    return total_flops[0] / 1e9


In [ ]:
# Cell 11 – Model Profiling: Params, GFLOPs, FPS across Input Sizes


import time
import torch
import torch.nn as nn
from torchinfo import summary as ti_summary


def get_dataset_native_size(pairs):
    """Read the first image from dataset pairs and return its (H, W)."""
    if not pairs:
        return None
    from PIL import Image as _PIL_Image
    img = _PIL_Image.open(pairs[0][0]).convert("RGB")
    w, h = img.size          # PIL returns (width, height)
    return (h, w)


def profile_model(model, input_sizes, device, n_warmup=10, n_runs=50):
    """
    Profile model across multiple input sizes.

    GFLOPs method
    -------------
    torchinfo.summary() counts MACs (multiply-accumulate ops) per registered
    nn.Module including:
      - Conv2d  (depthwise and standard)
      - Linear  (SE bottleneck squeeze/excite)
      - GroupNorm / BatchNorm
      - AdaptiveAvgPool2d  (SE global pool)
      - PixelShuffle  (decoder upsample)
      - F.interpolate  (final bilinear upsample)
    No JIT tracing is used, so there are no silently-zeroed unsupported ops.
    GFLOPs = total_mult_adds × 2 / 1e9  (1 MAC = 1 multiply + 1 add = 2 FLOPs)

    Args
    ----
    model       : nn.Module to profile (LightVesselNet)
    input_sizes : list of (H, W) tuples
    device      : torch.device
    n_warmup    : warm-up forward passes (discarded)
    n_runs      : timed forward passes for FPS estimation

    Returns
    -------
    list of dicts with keys:
        size, trainable_params, non_trainable_params,
        total_params, gflops, mmacs, fps, ms_per_image
    """
    model = model.to(device)
    model.eval()

    # ── Param counts (size-independent) ─────────────────────────
    trainable_params     = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params         = trainable_params + non_trainable_params

    results = []

    for h, w in input_sizes:
        # ── Accurate GFLOPs via torchinfo ────────────────────────
        stats  = ti_summary(
            model,
            input_size=(1, 3, h, w),
            device=device,
            verbose=0,       # suppress per-layer table; set to 1 to debug
            col_names=[],
        )
        gflops = stats.total_mult_adds * 2 / 1e9   # MACs × 2 = FLOPs
        mmacs  = stats.total_mult_adds / 1e6        # MMACs for reference

        # ── FPS / latency ────────────────────────────────────────
        dummy = torch.randn(1, 3, h, w, device=device)

        with torch.no_grad():
            for _ in range(n_warmup):
                model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()

        t_start = time.perf_counter()
        with torch.no_grad():
            for _ in range(n_runs):
                model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed    = time.perf_counter() - t_start
        ms_per_img = (elapsed / n_runs) * 1000.0
        fps        = 1000.0 / ms_per_img

        results.append({
            "size"                : f"{h}×{w}",
            "trainable_params"    : trainable_params,
            "non_trainable_params": non_trainable_params,
            "total_params"        : total_params,
            "gflops"              : gflops,
            "mmacs"               : mmacs,
            "fps"                 : fps,
            "ms_per_image"        : ms_per_img,
        })

    return results


def print_profile_table(results):
    """Pretty-print the profiling results as a table."""
    r0 = results[0]
    print("\n" + "═"*80)
    print("  Model Parameter Summary")
    print("═"*80)
    print(f"  Trainable params     : {r0['trainable_params']:>12,}  ({r0['trainable_params']/1e6:.4f} M)")
    print(f"  Non-trainable params : {r0['non_trainable_params']:>12,}  ({r0['non_trainable_params']/1e6:.4f} M)")
    print(f"  Total params         : {r0['total_params']:>12,}  ({r0['total_params']/1e6:.4f} M)")
    budget = "PASS (<0.2 M)" if r0["trainable_params"] < 200_000 else "FAIL (≥0.2 M)"
    print(f"  Param budget         : {budget}")
    print("═"*80)

    hdr = f"  {'Input Size':>14}  {'GFLOPs':>9}  {'MMACs':>9}  {'FPS':>9}  {'ms/img':>9}  {'<2 GFlop?':>11}"
    print("\n" + hdr)
    print("  " + "─"*(len(hdr)-2))
    for r in results:
        chk = "PASS" if r["gflops"] < 2.0 else "FAIL"
        print(f"  {r['size']:>14}  {r['gflops']:>9.4f}  {r['mmacs']:>9.1f}"
              f"  {r['fps']:>9.2f}  {r['ms_per_image']:>9.2f}  {chk:>11}")
    print("═"*80)
    print("  GFLOPs = MACs × 2.  Counted via torchinfo (Conv, Linear, Norm, Pool, Interp).")
    print("  FPS = batch-1 inference, n_runs=50, GPU-synchronized timing.")
    print("═"*80 + "\n")


# ── Determine sizes to profile ───────────────────────────────────
_all_pairs       = train_pairs + val_pairs + test_pairs
native_hw        = get_dataset_native_size(_all_pairs)
_benchmark_sizes = [(256, 256), (512, 512), (768, 768), (1024, 1024)]

if native_hw is not None and native_hw not in _benchmark_sizes:
    profile_sizes = [native_hw] + _benchmark_sizes
    print(f"  Detected native dataset resolution: {native_hw[0]}×{native_hw[1]}")
elif native_hw in _benchmark_sizes:
    profile_sizes = _benchmark_sizes
    print(f"  Native res {native_hw[0]}×{native_hw[1]} already in benchmark list.")
else:
    profile_sizes = _benchmark_sizes
    print("  Could not detect native resolution — using benchmark sizes only.")

# ── Run profiling ─────────────────────────────────────────────────
_prof_model = LightVesselNet(
    dropout_rate   = CFG_S1.DROPOUT_RATE,
    dropblock_size = CFG_S1.DROPBLOCK_SIZE,
).to(CFG.DEVICE)

from pathlib import Path as _Path
if _Path(CFG_S1.MODEL_SAVE_PATH).exists():
    _prof_model.load_state_dict(
        torch.load(CFG_S1.MODEL_SAVE_PATH, map_location=CFG.DEVICE)
    )
    print("  Loaded best weights for profiling.")
else:
    print("  No saved weights found — profiling with random-init weights.")

profile_results = profile_model(_prof_model, profile_sizes, CFG.DEVICE)
print_profile_table(profile_results)
del _prof_model

In [ ]:
# Cell 12a – Training Loop 

import albumentations as A
from albumentations.pytorch import ToTensorV2

# norm_inf: no resize — images are passed at native resolution
norm_inf = A.Compose([
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()])


def _preprocess_image(image):
    """Apply CLAHE pre-processing + normalise at native resolution → (1,3,H,W) tensor."""
    image = apply_preprocessing(image)
    t = norm_inf(image=image)["image"].unsqueeze(0).to(CFG.DEVICE)
    return t


def predict_image_s1(model, image, use_tta=None):
    """
    Whole-image inference with 5-view TTA (orig + H/V flip + rot90/270).
    Returns: (soft_prob, binary_map) at native image resolution (no resize).
    """
    if use_tta is None:
        use_tta = CFG_S1.USE_TTA
    model.eval()

    def _infer(img_np):
        t = _preprocess_image(img_np)
        with torch.no_grad():
            out    = model(t)
            logits = out[0] if isinstance(out, tuple) else out
            return torch.sigmoid(logits).squeeze().cpu().numpy().astype(np.float32)

    soft = _infer(image)
    if use_tta:
        views = [
            np.fliplr(_infer(np.fliplr(image))),                    # H-flip
            np.flipud(_infer(np.flipud(image))),                    # V-flip
            np.rot90(_infer(np.rot90(image, k=1)), k=-1),           # rot90
            np.rot90(_infer(np.rot90(image, k=3)), k=-3 % 4),      # rot270
        ]
        soft = (soft + sum(views)) / (1 + len(views))

    binary = (soft >= 0.5).astype(np.uint8)
    return soft.astype(np.float32), binary

def predict_with_uncertainty(model, image, n_samples=None):
    """MC Dropout inference with TTA. Returns (mean_prob, variance)."""
    if n_samples is None:
        n_samples = CFG_S1.MC_SAMPLES
    image_proc = apply_preprocessing(image)
    all_maps   = []
    model.train()   # keep dropout active

    def _infer_mc(img_np):
        t = norm_inf(image=img_np)["image"].unsqueeze(0).to(CFG.DEVICE)
        with torch.no_grad():
            out    = model(t)
            logits = out[0] if isinstance(out, tuple) else out
            return torch.sigmoid(logits).squeeze().cpu().numpy().astype(np.float32)

    for _ in range(n_samples):
        soft = _infer_mc(image_proc)
        if CFG_S1.USE_TTA:
            soft_hf = np.fliplr(_infer_mc(np.fliplr(image_proc)))
            soft_vf = np.flipud(_infer_mc(np.flipud(image_proc)))
            soft = (soft + soft_hf + soft_vf) / 3.0
        all_maps.append(soft)

    model.eval()
    stack       = np.stack(all_maps, axis=0)
    mean_prob   = stack.mean(axis=0)
    uncertainty = stack.var(axis=0)
    return mean_prob.astype(np.float32), uncertainty.astype(np.float32)
    
def find_best_threshold(model, val_pairs):
    all_probs, all_gts = [], []
    model.eval()
    for img_path, mask_path in val_pairs:
        image   = np.array(Image.open(img_path).convert("RGB"))
        gt_mask = (np.array(Image.open(mask_path).convert("L")) > 127).astype(np.uint8)
        soft, _ = predict_image_s1(model, image, use_tta=False)
        gt_rsz  = cv2.resize(gt_mask, (soft.shape[1], soft.shape[0]),
                              interpolation=cv2.INTER_NEAREST)
        all_probs.append(soft.ravel()); all_gts.append(gt_rsz.ravel())
    probs = np.concatenate(all_probs); gts = np.concatenate(all_gts)
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.3, 0.7, 0.02):
        pred = (probs >= t).astype(np.uint8)
        tp   = ((pred==1)&(gts==1)).sum()
        fp   = ((pred==1)&(gts==0)).sum()
        fn   = ((pred==0)&(gts==1)).sum()
        f1   = 2*tp / (2*tp+fp+fn+1e-8)
        if f1 > best_f1: best_f1, best_t = f1, t
    return best_t, best_f1


def validate_s1(model, val_pairs, criterion):
    model.eval()
    all_metrics = []
    for img_path, mask_path in val_pairs:
        image   = np.array(Image.open(img_path).convert("RGB"))
        gt_mask = (np.array(Image.open(mask_path).convert("L")) > 127).astype(np.uint8)
        soft, _ = predict_image_s1(model, image, use_tta=False)
        gt_rsz  = cv2.resize(gt_mask, (soft.shape[1], soft.shape[0]),
                              interpolation=cv2.INTER_NEAREST)
        all_metrics.append(compute_metrics(soft, gt_rsz))
    avg = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0]}
    return 0.0, avg


def get_lr(optimizer): return optimizer.param_groups[0]["lr"]

In [ ]:
# Cell 12b – Training Loop 

def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler, epoch):
    model.train()
    running_loss = 0.0
    for batch_idx, batch in enumerate(loader):
        imgs  = batch["image"].to(CFG.DEVICE)
        masks = batch["mask" ].to(CFG.DEVICE)

        r = np.random.rand()
        if CFG_S1.CUTMIX_PROB > 0 and r < CFG_S1.CUTMIX_PROB:
            imgs, masks = cutmix_batch(imgs, masks, prob=1.0)
        elif CFG_S1.MIXUP_ALPHA > 0 and r < CFG_S1.CUTMIX_PROB + CFG_S1.MIXUP_ALPHA:
            imgs, masks = mixup_batch(imgs, masks, alpha=CFG_S1.MIXUP_ALPHA)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            main_logits, aux3, aux2 = model(imgs, return_aux=True)
            loss = criterion(main_logits, aux3, aux2, masks)

        if torch.isnan(loss):
            print(f"  ⚠️  NaN loss at batch {batch_idx} – skipping"); continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG_S1.GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        scheduler.step()
        running_loss += loss.item()
    return running_loss / max(len(loader), 1)

def train_stage1():
    global model_s1, BEST_THRESHOLD
    model_s1 = LightVesselNet(
        dropout_rate   = CFG_S1.DROPOUT_RATE,
        dropblock_size = CFG_S1.DROPBLOCK_SIZE,
    ).to(CFG.DEVICE)
    optimizer = torch.optim.AdamW(model_s1.parameters(),
                                  lr=CFG_S1.LR, weight_decay=CFG_S1.WEIGHT_DECAY)
    total_steps = CFG_S1.EPOCHS * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr           = CFG_S1.LR,
        total_steps      = total_steps,
        pct_start        = 0.1,
        anneal_strategy  = "cos",
        div_factor       = 25.0,
        final_div_factor = 1e4,
    )
    scaler           = torch.amp.GradScaler("cuda")
    history          = {"train_loss": [], "val_loss": [], "val_dice": [],
                        "val_iou": [], "val_auc": [], "val_composite": []}
    best_composite   = 0.0
    patience_counter = 0
    VAL_FREQ         = 2
    print("\n" + "="*92)
    print(f"{'Epoch':>6} {'TrainLoss':>10} {'ValLoss':>9} {'Dice':>7} {'IoU':>7} "
          f"{'SE':>7} {'Mean':>7} {'AUC':>7} {'LR':>10} {'Pat':>4}")
    print("="*92)
    last_v_m = {"f1": 0.0, "iou": 0.0, "auc": 0.0, "se": 0.0, "composite": 0.0}
    for epoch in range(1, CFG_S1.EPOCHS + 1):
        # pass scheduler into train_one_epoch so it can step per batch
        t_loss = train_one_epoch(model_s1, train_loader, optimizer, scheduler, criterion, scaler, epoch)
        # scheduler.step() removed from here — now inside train_one_epoch
        run_val = val_pairs and (epoch % VAL_FREQ == 0 or epoch == 1
                                 or epoch == CFG_S1.EPOCHS)
        if run_val:
            _, v_m   = validate_s1(model_s1, val_pairs, criterion)
            last_v_m = v_m
            last_v_m["composite"] = (last_v_m["f1"] + last_v_m.get("se", 0.0)) / 2.0
        lr = get_lr(optimizer)
        history["train_loss"].append(t_loss)
        if run_val:
            history["val_loss"].append(last_v_m["f1"])
            history["val_dice"].append(last_v_m["f1"])
            history["val_iou"].append(last_v_m["iou"])
            history["val_auc"].append(last_v_m["auc"])
            history["val_composite"].append(last_v_m["composite"])
        if run_val:
            print(f"{epoch:>6} {t_loss:>10.4f} {'val':>9} {last_v_m['f1']:>7.4f} "
                  f"{last_v_m['iou']:>7.4f} {last_v_m.get('se', 0):>7.4f} "
                  f"{last_v_m['composite']:>7.4f} {last_v_m['auc']:>7.4f} "
                  f"{lr:>10.2e} {patience_counter:>4}")
        else:
            print(f"{epoch:>6} {t_loss:>10.4f} {'---':>9} {'---':>7} "
                  f"{'---':>7} {'---':>7} {'---':>7} {'---':>7} {lr:>10.2e} {'':>4}")
        if run_val:
            if last_v_m["composite"] > best_composite:
                best_composite   = last_v_m["composite"]
                patience_counter = 0
                torch.save(model_s1.state_dict(), CFG_S1.MODEL_SAVE_PATH)
                print(f"  💾 Saved best (Dice={last_v_m['f1']:.4f}, SE={last_v_m.get('se', 0):.4f}, Mean={best_composite:.4f})")
            else:
                patience_counter += 1
                if patience_counter >= CFG_S1.PATIENCE:
                    print(f"  🛑 Early stopping at epoch {epoch}")
                    break
    if val_pairs and Path(CFG_S1.MODEL_SAVE_PATH).exists():
        model_s1.load_state_dict(torch.load(CFG_S1.MODEL_SAVE_PATH,
                                            map_location=CFG.DEVICE))
        BEST_THRESHOLD, _ = find_best_threshold(model_s1, val_pairs)
        print(f"  🎯 Optimal threshold from val set: τ={BEST_THRESHOLD:.2f}")
    else:
        BEST_THRESHOLD = 0.5
    return history

BEST_THRESHOLD = 0.5
print("Training functions defined.")


In [ ]:
# Cell 13 – Run Training
criterion = VesselLoss() 

if train_pairs:
    history_s1 = train_stage1()
else:
    print("⚠️  No training pairs – skipping.")
    history_s1 = None


In [ ]:
# Cell 14 – Final Test-Set Evaluation (Resolution-Aware)
# ─────────────────────────────────────────────────────
import time
import numpy as np
import torch
import cv2
from pathlib import Path
from PIL import Image

def evaluate_test_set(model_s1, test_pairs, threshold=None):
    if threshold is None:
        threshold = BEST_THRESHOLD

    model_s1.eval()

    print(f"  Threshold τ                           : {threshold:.2f}  (val-set Dice-optimal)")
    print(f"  TTA                                   : {CFG_S1.USE_TTA}")
    print(f"  MC samples                            : {CFG_S1.MC_SAMPLES}")

    results = []
    all_metrics, all_time = [], []

    hdr = (f"{'Image':>22} {'SE':>7} {'SP':>7} {'ACC':>7} "
           f"{'AUC':>7} {'PR':>7} {'F1':>7} {'IoU':>7} {'ms':>8}")
    sep = "=" * len(hdr)

    print("\n" + sep)
    print(hdr)
    print(sep)

    for img_path, mask_path in test_pairs:
        # ── Load data ────────────────────────────────
        image   = np.array(Image.open(img_path).convert("RGB"))
        gt_mask = (np.array(Image.open(mask_path).convert("L")) > 127).astype(np.uint8)

        # ── Inference timing ─────────────────────────
        t0 = time.perf_counter()
        s1_prob, s1_unc = predict_with_uncertainty(
            model_s1,
            image,
            n_samples=CFG_S1.MC_SAMPLES
        )
        elapsed_ms = (time.perf_counter() - t0) * 1000.0

        # ── Resize GT to match prediction ────────────
        gt_rsz = cv2.resize(
            gt_mask,
            (s1_prob.shape[1], s1_prob.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

        # ── Metrics ─────────────────────────────────
        m = compute_metrics(s1_prob, gt_rsz, threshold=threshold)

        all_metrics.append(m)
        all_time.append(elapsed_ms)

        print(f"{img_path.name:>22} "
              f"{m['se']:>7.4f} {m['sp']:>7.4f} {m['acc']:>7.4f} "
              f"{m['auc']:>7.4f} {m['pr']:>7.4f} {m['f1']:>7.4f} "
              f"{m['iou']:>7.4f} {elapsed_ms:>8.1f}")

        results.append({
            "name": img_path.name,
            "image": image,
            "gt": gt_rsz,
            "original_gt": gt_mask,
            "s1_soft": s1_prob,
            "s1_unc": s1_unc,
            "metrics": m,
            "ms": elapsed_ms
        })

    # ── Summary ─────────────────────────────────────
    print(sep)
    print(f"\n  Total test images evaluated: {len(test_pairs)}")

    print("\nAggregate Summary (mean ± std):")
    print("-" * 55)

    for label, key in [("SE","se"), ("SP","sp"), ("ACC","acc"),
                       ("AUC","auc"), ("PR","pr"), ("F1","f1"), ("IoU","iou")]:
        vals = [m[key] for m in all_metrics]
        print(f"  {label:<6}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

    print("-" * 55)

    return results


# ── Run evaluation ───────────────────────────────────
if test_pairs and Path(CFG_S1.MODEL_SAVE_PATH).exists():
    model_s1.load_state_dict(
        torch.load(CFG_S1.MODEL_SAVE_PATH, map_location=CFG.DEVICE)
    )
    print("Best weights loaded.")
    test_results = evaluate_test_set(model_s1, test_pairs)
else:
    print("Skipping test evaluation (no test pairs or saved weights).")
    test_results = []

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

def compute_confusion_map(gt, pred):
    """
    TP = Green, TN = Black, FP = Red, FN = Blue
    """
    gt = gt.astype(np.uint8)
    pred = pred.astype(np.uint8)

    tp = (gt == 1) & (pred == 1)
    tn = (gt == 0) & (pred == 0)
    fp = (gt == 0) & (pred == 1)
    fn = (gt == 1) & (pred == 0)

    h, w = gt.shape
    cmap = np.zeros((h, w, 3), dtype=np.float32)

    cmap[tp] = [0, 1, 0]   # TP → green
    cmap[tn] = [0, 0, 0]   # TN → black
    cmap[fp] = [1, 0, 0]   # FP → red
    cmap[fn] = [0, 0, 1]   # FN → blue

    return cmap


def visualize_test_results(
    test_results,
    num_samples=2,
    random_select=False,
    save_path="test_visualizations/combined_results.jpg"
):
    """
    Create ONE combined visualization image and save it.
    """

    if not test_results:
        print("⚠️ No test results available.")
        return

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # ── Select samples ──────────────────────────────
    if random_select:
        idxs = np.random.choice(
            len(test_results),
            min(num_samples, len(test_results)),
            replace=False
        )
    else:
        idxs = list(range(min(num_samples, len(test_results))))

    n = len(idxs)

    # rows = samples, cols = 6 views
    fig, axes = plt.subplots(n, 6, figsize=(26, 5 * n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(idxs):

        r = test_results[idx]

        image       = r["image"]
        gt          = r["gt"]
        pred_prob   = r["s1_soft"]
        uncertainty = r["s1_unc"]

        pred_mask = (pred_prob >= BEST_THRESHOLD).astype(np.uint8)
        conf_map  = compute_confusion_map(gt, pred_mask)

        # 1. Image
        axes[row, 0].imshow(image)
        axes[row, 0].set_title("Image")
        axes[row, 0].axis("off")

        # 2. GT
        axes[row, 1].imshow(gt, cmap="gray")
        axes[row, 1].set_title("GT")
        axes[row, 1].axis("off")

        # 3. Probability
        im2 = axes[row, 2].imshow(pred_prob, cmap="jet")
        axes[row, 2].set_title("Prob.")
        axes[row, 2].axis("off")

        # 4. Mask
        axes[row, 3].imshow(pred_mask, cmap="gray")
        axes[row, 3].set_title("Mask")
        axes[row, 3].axis("off")

        # 5. Uncertainty
        im4 = axes[row, 4].imshow(uncertainty, cmap="hot")
        axes[row, 4].set_title("Uncertainty")
        axes[row, 4].axis("off")

        # 6. TP/TN/FP/FN
        axes[row, 5].imshow(conf_map)
        axes[row, 5].set_title("TP/TN/FP/FN")
        axes[row, 5].axis("off")

        # Legend (only on last column)
        axes[row, 5].text(
            0.02, 0.02,
            "Green=TP | Black=TN\nRed=FP | Blue=FN",
            transform=axes[row, 5].transAxes,
            fontsize=10,
            color="white",
            bbox=dict(facecolor="black", alpha=0.5)
        )

    plt.tight_layout()

    # ── SAVE SINGLE COMBINED IMAGE ─────────────────
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"✅ Combined visualization saved at: {save_path}")


# ── RUN ──────────────────────────────
visualize_test_results(
    test_results,
    num_samples=2,
    random_select=False
)